In [6]:
import numpy as np
import random
import networkx as nx

def generate_dsbm(N, M, min_rate, max_rate, alpha_range, beta_range):
    # Step 1: Generate block sizes randomly between 0.01% and 0.9% of N
    min_block_size = int(min_rate * N)
    max_block_size = int(max_rate * N)
    
    # Random block sizes
    block_sizes = np.random.randint(min_block_size, max_block_size, size=M)
    # Normalize block sizes to ensure their sum equals N
    block_sizes = (block_sizes / block_sizes.sum() * N).astype(int)
    # Adjust any discrepancy caused by rounding errors
    difference = N - block_sizes.sum()
    block_sizes[0] += difference
    

    edge_list = []
    block_assignment = {}
    
    # Step 2: Assign blocks to nodes and keep track of block indices
    block_indices = []
    start_idx = 0
    for block_id, size in enumerate(block_sizes):
        block_indices.append(list(range(start_idx, start_idx + size))) 
        # a list of lists, each list contains the node id in this block
        for node in range(start_idx, start_idx + size):
            block_assignment[node] = block_id  # Assign the node to the block
        start_idx += size
    
    # Step 3: Fill the edge list based on alpha (intra-block) and beta (inter-block)
    for i in range(M): # can be faster?
        # Intra-block edges with probability alpha
        alpha = random.uniform(*alpha_range)
        for u in block_indices[i]:
            for v in block_indices[i]:
                if u != v:  # Avoid self-loops
                    if np.random.rand() < alpha:
                        edge_list.append((u, v))
        
        # Inter-block edges with probability beta
        for j in range(i + 1, M):
            beta = random.uniform(*beta_range)
            for u in block_indices[i]:
                for v in block_indices[j]:
                    if np.random.rand() < beta:
                        edge_list.append((u, v))  # Edge from block i to block j
                    if np.random.rand() < beta:
                        edge_list.append((v, u))  # Edge from block j to block i
    
    return edge_list, block_assignment

def adjust_edges(edge_list, block_assignment, N, X, Y):
    # Step 1: Initialize the adjacency list to track out-edges for each node
    out_edges = {v: [] for v in range(N)}
    for u, v in edge_list:
        out_edges[u].append(v)
    
    # Step 2: Iterate over each node
    for v in range(N):
        # Step 3: Pick random values x and y (desired out-edges)
        x = random.randint(0, X)
        y = random.randint(0, Y)
        
        # Step 4: Count current inter-block (i) and intra-block (j) out-edges
        intra_neighbors = [neighbor for neighbor in out_edges[v] if block_assignment[neighbor] == block_assignment[v]]
        inter_neighbors = [neighbor for neighbor in out_edges[v] if block_assignment[neighbor] != block_assignment[v]]
        i = len(intra_neighbors)
        j = len(inter_neighbors)
        
        # Step 5: Adjust intra-block edges if i < x
        while i < x:
            if i == 0:
                break
            # Pick a random neighbor from the same block
            neighbor = random.choice(intra_neighbors)
            edge_list.append((v, neighbor))
            out_edges[v].append(neighbor)
            i += 1
        
        # Step 6: Adjust inter-block edges if j < y
        while j < y:
            if j==0:
                break
            neighbor = random.choice(inter_neighbors)     
            edge_list.append((v, neighbor))
            out_edges[v].append(neighbor)
            j += 1
    
    return edge_list



In [9]:
# Example usage
N = 50000   # Total number of nodes
M = 5000      # Number of blocks (modules)
alpha = (0.0003,0.05)  # Intra-block connection probability
beta = (0.0001,0.0002)   # Inter-block connection probability
X = 6  # Maximum intra-block edges for each node
Y = 3   # Maximum inter-block edges for each node

# Generate initial DSBM network
edge_list, block_assignment, block_indices = generate_dsbm(N, M, 0.0001,0.009,alpha, beta)


In [10]:
edge_list

[(0, 101),
 (0, 527),
 (0, 549),
 (0, 678),
 (0, 845),
 (0, 1531),
 (0, 2028),
 (1, 62),
 (1, 129),
 (1, 232),
 (1, 256),
 (1, 622),
 (1, 736),
 (1, 1351),
 (1, 1739),
 (1, 1817),
 (1, 1841),
 (1, 1938),
 (2, 34),
 (2, 47),
 (2, 308),
 (2, 642),
 (2, 1461),
 (2, 1951),
 (2, 2182),
 (2, 2259),
 (3, 143),
 (3, 266),
 (3, 362),
 (3, 531),
 (3, 623),
 (3, 808),
 (3, 1312),
 (3, 1608),
 (3, 1801),
 (3, 2164),
 (3, 2388),
 (4, 529),
 (4, 689),
 (4, 1293),
 (4, 1365),
 (4, 1540),
 (4, 1544),
 (4, 1844),
 (5, 732),
 (5, 1108),
 (5, 1180),
 (5, 1562),
 (5, 1677),
 (5, 1769),
 (5, 2139),
 (5, 2327),
 (5, 2463),
 (6, 184),
 (6, 303),
 (6, 916),
 (6, 959),
 (6, 1079),
 (6, 1317),
 (6, 1701),
 (6, 1786),
 (6, 1930),
 (6, 2118),
 (7, 310),
 (7, 379),
 (7, 431),
 (7, 1045),
 (7, 1198),
 (7, 1493),
 (7, 1557),
 (7, 2372),
 (7, 2454),
 (8, 270),
 (8, 276),
 (8, 290),
 (8, 620),
 (8, 746),
 (8, 1275),
 (8, 1744),
 (8, 1966),
 (8, 2228),
 (9, 82),
 (9, 90),
 (9, 295),
 (9, 334),
 (9, 368),
 (9, 398),
 (9

In [ ]:
# Adjust edges according to the x, y conditions
final_edge_list = adjust_edges(edge_list, block_assignment, block_indices, N, X, Y)

# Print the results
print("Final Edge List (first 10 edges):\n", final_edge_list[:10])
print("Block Assignment:\n", {k: block_assignment[k] for k in list(block_assignment.keys())[:10]})

G = nx.MultiDiGraph()

# Add edges to the graph from the edge list
G.add_edges_from(final_edge_list)